# 15. Multilayer Perceptron (MLP)

Feedforward neural net for tabular binary classification.
Uses ReLU hidden layers; sklearn handles backprop / gradient descent.
Not a CNN — convolutions are for grid-like data (images).

Preprocessing: `scale=True` (important for NNs), `dense=True` (dense matrix after one-hot).
Fixed architecture for a fair comparison (no big grid search).

In [1]:
import sys
sys.path.append('..')

from sklearn.neural_network import MLPClassifier

from src.data import load_adult
from src.preprocess import make_model_pipeline, split_features_target
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult('train'))
X_test, y_test = split_features_target(load_adult('test'))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [2]:
candidates = [300, 500]
selection_metric = "f1"

cv_by_max_iter = {}
best_max_iter = None
best_score = -1.0

for max_iter in candidates:
    model = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        max_iter=max_iter,
        random_state=42,
    )
    pipe = make_model_pipeline(model, scale=True, dense=True)
    cv = cross_val_scores(pipe, X, y)
    cv_by_max_iter[max_iter] = cv
    score = cv[selection_metric]["mean"]
    print(f"max_iter={max_iter}  CV {selection_metric}={score:.4f}  (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_max_iter = max_iter

print("selected max_iter:", best_max_iter)

/Users/kang/Desktop/ML project/Estimating-Income/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/kang/Desktop/ML project/Estimating-Income/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


max_iter=300  CV f1=0.6467  (±0.0172)
max_iter=500  CV f1=0.6431  (±0.0155)
selected max_iter: 300


In [3]:
final_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=best_max_iter,
    random_state=42,
)
cv = cv_by_max_iter[best_max_iter]
print("CV:", cv)

pipe2 = make_model_pipeline(final_mlp, scale=True, dense=True)
test = evaluate_on_test(pipe2, X, y, X_test, y_test)
print("Test:", test["metrics"])
print("Confusion matrix:", test["confusion_matrix"])

path = save_results(
    "mlp",
    cv,
    test,
    extra={
        "hidden_layer_sizes": (64, 32),
        "activation": "relu",
        "solver": "adam",
        "max_iter": best_max_iter,
        "scale": True,
        "dense": True,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_max_iter": {
            str(m): {k: v["mean"] for k, v in summary.items()}
            for m, summary in cv_by_max_iter.items()
        },
    },
)
print("Saved to:", path)

CV: {'accuracy': {'mean': 0.8358463520888672, 'std': 0.005703842836393429}, 'balanced_accuracy': {'mean': 0.7640529492812179, 'std': 0.014970674857920316}, 'precision': {'mean': 0.672720891719301, 'std': 0.02371069685022059}, 'recall': {'mean': 0.6255573548731156, 'std': 0.04381171687365762}, 'f1': {'mean': 0.6466997111699949, 'std': 0.01722728911512099}, 'roc_auc': {'mean': 0.884980755641039, 'std': 0.0034457270009983065}, 'average_precision': {'mean': 0.7236491079810319, 'std': 0.008670593540021974}}
Test: {'accuracy': 0.8336097291321172, 'balanced_accuracy': 0.7604202382811838, 'precision': 0.6559670781893004, 'recall': 0.6216848673946958, 'f1': 0.6383660392470966, 'roc_auc': 0.8817228161583238, 'average_precision': 0.7109425713282309}
Confusion matrix: [[11181, 1254], [1455, 2391]]
Saved to: /Users/kang/Desktop/ML project/Estimating-Income/results/mlp.json


/Users/kang/Desktop/ML project/Estimating-Income/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
